<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 3 (Miércoles) — Clasificación 2 — Naive Bayes y regresión logística**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# ✉️ Naive Bayes: teoría y construcción desde cero (detector de spam)

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Entender el teorema de Bayes y el supuesto de independencia que hace "ingenuo" al clasificador. |
| 🔍 Comprensión de los datos | Cargamos `spambase` (4 601 correos, 57 variables de frecuencia de palabras y símbolos). |
| 🧹 Preparación de los datos | Estandarizamos las variables y dividimos 70/30. |
| 🤖 Modelado | Programamos Naive Bayes gaussiano desde cero (priori, medias, varianzas, log-posterior). |
| 📊 Evaluación | Evaluamos con exactitud, curva ROC (usando probabilidades), matriz de confusión, sensibilidad y especificidad. |
| 🚀 Despliegue y conclusiones | Comparamos con `scikit-learn` y con las variantes multinomial y Bernoulli. |

> 🎯 **Analogía:** un portero que revisa cada correo con una lista de pistas ("free", "$", "!!!") y multiplica qué tan típica es cada pista en el spam frente al correo normal.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 👉 **3** | Miércoles | **Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales** |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('spambase.csv', 'dia_4')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Entender el teorema de Bayes y el supuesto de independencia que hace "ingenuo" al clasificador.

📚 Lectura recomendada: [3] [4]

# **Naive Bayes**

Es un clasificador probabilístico fundamentado en el teorema de Bayes y algunas hipótesis simplificadoras adicionales. Es a causa de estas simplificaciones, que se suelen resumir en la hipótesis de independencia entre las variables clasificadoras.

![Naive Bayes gaussiano: cada clase se modela con una distribución normal por variable](https://raw.githubusercontent.com/jakevdp/PythonDataScienceHandbook/master/notebooks/figures/05.05-gaussian-NB.png)
*Figura: Naive Bayes gaussiano: cada clase se modela con una distribución normal por variable. [12]*


Una ventaja del clasificador ingenuo de Bayes es que solo se requiere una pequeña cantidad de datos de entrenamiento para estimar los parámetros necesarios para la clasificación (las medidas y las varianzas de las variables).

Solo es necesario determinar las varianzas de las variables de cada clase y no toda la matriz de covarianza. Para otros modelos de probabilidad, los clasificadores ingenuos de Bayes  se pueden entrenar en entornos de aprendizaje supervisado.

### **Teorema de Bayes**

El teorema de bayes esta expresado por la siguiente ecuación:

$$
P(H \mid D)=\frac{P(D \mid H) P(H)}{P(D)}
$$


**P(H)** es la probabilidad a priori, la forma de introducir conocimiento previo sobre los valores que puede tomar la hipótesis.

**P(D|H)** es el likelihood de una hipótesis H dados los datos D,  es decir, la probabilidad de obtener D dado que H es verdadera.

**P(D)** es el likelihood marginal o evidencia, es la probabilidad de observar los datos D promediado sobre todas las posibles hipótesis H.

**P(H|D)** es el a posteriori, la distribución de probabilidad final para la hipótesis. Es la consecuencia lógica de haber usado un conjunto de datos, un likelihood y un a priori.


Sobre una variable dependiente **H**, con un pequeño número de clases, la variable esta condicionada por varias variables independientes D = {d1, d2, …, dn} las cuales, dado el supuesto de independencia condicional de bayes, se asume que cada **di** es independiente de cualquier otro **dj** para **i** diferente de **j** y la podemos expresar en términos simples de la siguiente forma:

$$$$
$$
\text { Posterior }=\frac{\text { Probabilidad } x \text { Anterior }}{\text { Evidencia }}
$$

$$$$

La formula nos indica la probabilidad de que una hipótesis **H** sea verdadera si algún evento **D** ha sucedido. Esto es importante dado que, normalmente obtenemos la probabilidad de los efectos dadas las causas, pero el teorema de bayes nos indica la probabilidad de las causas dados los efectos.

Por ejemplo, podemos saber cual es el porcentaje de pacientes con gripe que tienen fiebre, pero lo que realmente queremos saber es la probabilidad de que un paciente con fiebre tenga gripe.


Tenemos un conjunto de datos de personas que camina o conducen hacia su trabajo, en relación a su edad y a su salario, por ejemplo.



Personas que camina o conducen hacia su trabajo en relación a la edad y el salario
Si ahora tenemos la edad y el salario de una nueva persona, queremos clasificarla, de acuerdo a esos datos, si es de las personas que caminan o de las que conducen.



Una nueva persona de la que tenemos su edad y su salario, es de las que conduce o camina?
$$$$
$$P($ Camina $\mid X)=\frac{P(X \mid \text { Camina }) P(\text { Camina })}{P(X)}$$
$$$$
$$P($ Conduce $\mid X)=\frac{P(X \mid \text { Conduce }) P(\text { Conduce })}{P(X)}$$

$$$$
X: representa el nuevo punto (Edad, Salario)
$$$$

Teorema de bayes para clasificar a una nueva persona en base a su edad y su salario




P(Camina) es el Número de personas que caminan entre el total de observaciones



P(X) es el Número de observaciones similares al nuevo punto, entre el total de observaciones



P(X|Camina) es el número de observaciones similares entre los que caminan entre el total de los que caminan



Aplicando los valores a la formula del teorema



También para los que conducen
Si ahora comparamos los que caminan contra los que conducen tenemos que:

P(Camina|X) > P(Conduce |X)
0.75 > 0.25

Entonces, este nuevo punto que representa la edad y el salario de una persona nueva, será clasificado en el grupo de los que caminan.



El nuevo punto ha sido clasificado entre las personas que caminan

## **Ejemplo**

Tenemos dos máquinas (m1 y m2) que fabrican la misma herramienta



Dos máquinas que fabrican la misma herramienta
De todas las herramientas que fabrica cada una de las máquinas, algunas se producen con defectos.



Herramientas producidas por las máquinas m1 y m2, algunas con defectos (color negro)
Si consideramos que la máquina 1 produce 30 llaves por hora y la máquina 2 produce 20 llaves por hora, de todas las partes producidas se observa que el 1% están defectuosas y de todas las llaves defectuosas el 50% provienen de la máquina 1 y el 50% de la máquina 2.

¿Cuál es la probabilidad de que una pieza defectuosa haya sido producida por la máquina 2?

Si M1: 30 llaves/hora, M2: 20 llaves/hora
de las defectuosas 50%  son de M1 y 50% de M2

P(M1) = 30/50 = 0.6
P(M2) = 20/50 = 0.4
P(Defecto) = 1%
P(M1 | Defecto) = 50%
P(M2 | Defecto) = 50%

Lo que deseamos conocer es entonces:
P(Defecto | M2) = ?

Aplicando el Teorema de Bayes

$$
P(\text { Defecto } \mid M 2)=\frac{P(\text { M2 } \mid \text { Defecto }) P(\text { Defecto })}{P(M 2)}
$$

Teorema de bayes para las máquinas que producen llaves

$$
P(\text { Defecto } \mid M 2)=\frac{0.5 * 0.01}{0.4}=0.0125
$$

Sustituyendo el valor de las probabilidades
La probabilidad de que una pieza fabricada por la máquina 2 sea defectuosa es del 1.25 %

En una producción de 1,000 piezas, entonces 400 provienen dela máquina 2 y si el 1% esta defectuosa habrá 10 piezas defectuosas. de esas 10 piezas el 50% son la máquina 2, es decir 5 piezas, podemos comprobar que el porcentaje de piezas defectuosas de la máquina 2 es 5/400 = 0.0125

**Ejemplo con piezas defectuosas (aplicación del teorema de Bayes).** Con 1 000 piezas producidas, la máquina 1 hace 600 y la máquina 2 hace 400; el 1 % (10 piezas) sale defectuoso, y de esas 10 la mitad viene de cada máquina:

| | Máquina 1 | Máquina 2 | Total |
|---|---|---|---|
| Piezas producidas | 600 | 400 | 1 000 |
| Piezas defectuosas | 5 | 5 | 10 |
| % defectuosas dentro de la máquina | 0.83 % | **1.25 %** | 1 % |

La máquina 2 es la que más defectos produce en proporción: $P(\text{defecto}\mid M2)=\dfrac{5}{400}=1.25\,\%$, justo lo que da el teorema de Bayes [7].

### **Codigo desde 0 Naive Bayes**

---
## 🔍 Fase 2 · Comprensión de los datos — cargar spambase

Cargamos `spambase` (4 601 correos, 57 variables de frecuencia de palabras y símbolos).

📚 Lectura recomendada: [6]

In [ ]:
import warnings;
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
import plotly.express as px
import plotly.figure_factory as ff

In [ ]:
data=pd.read_csv('sample_data/spambase.csv')
#data['spam'] = data['spam'].map({1: "spam", 0: "not_spam"})
#data['spam'] = data['spam'].astype('category')

In [ ]:
data.head()

In [ ]:
data.info()

---
## 🤖 Fase 4 · Modelado — clasificador Naive Bayes programado

Programamos Naive Bayes gaussiano desde cero (priori, medias, varianzas, log-posterior).

📚 Lectura recomendada: [7] [5]

In [ ]:
class NaiveBayesClassifier():
    '''
    Bayes Theorem form
    P(y|X) = P(X|y) * P(y) / P(X)
    '''
    def calc_prior(self, features, target):
        '''
        prior probability P(y)
        calculate prior probabilities
        '''
        self.prior = (features.groupby(target).size() / self.rows).to_numpy()

        return self.prior

    def calc_statistics(self, features, target):
        '''
        calculate mean, variance for each column and convert to numpy array
        '''
        self.mean = features.groupby(target).mean().to_numpy()   # media por clase y por variable (en pandas/NumPy recientes np.mean(df) ya no devuelve una media por columna)
        self.var = features.groupby(target).var(ddof=0).to_numpy() + 1e-9 * features.to_numpy().var(axis=0).max()   # suavizado de varianza: evita dividir por cero

        return self.mean, self.var

    def gaussian_density(self, class_idx, x):
        '''
        calculate probability from gaussian density function (normally distributed)
        we will assume that probability of specific target value given specific class is normally distributed

        probability density function derived from wikipedia:
        (1/√(2πσ²)) · exp(−(x−μ)² / (2σ²)), donde μ es la media y σ² la varianza
        '''
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))   # corregido: antes tenía un factor 1/2 de más
#         numerator = np.exp(-((x-mean)**2 / (2 * var)))
        denominator = np.sqrt(2 * np.pi * var)
        prob = numerator / denominator
        return prob

    def calc_posterior(self, x):
        posteriors = []

        # calculate posterior probability for each class
        for i in range(self.count):
            prior = np.log(self.prior[i]) ## use the log to make it more numerically stable
            conditional = np.sum(self.log_gaussian_density(i, x)) # use the log to make it more numerically stable
            posterior = prior + conditional
            posteriors.append(posterior)
        # return class with highest posterior probability
        return self.classes[np.argmax(posteriors)]


    def fit(self, features, target):
        self.classes = np.unique(target)
        self.count = len(self.classes)
        self.feature_nums = features.shape[1]
        self.rows = features.shape[0]

        self.calc_statistics(features, target)
        self.calc_prior(features, target)

    def predict(self, features):
        preds = [self.calc_posterior(f) for f in features.to_numpy()]
        return preds

    def log_gaussian_density(self, class_idx, x):
        '''logaritmo de la densidad normal (estable: evita log(0) cuando la densidad se vuelve ~0)'''
        mean, var = self.mean[class_idx], self.var[class_idx]
        return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)

    def predict_proba(self, features):
        '''Probabilidad a posteriori de cada clase (normalizada con log-sum-exp)'''
        salida = []
        for f in features.to_numpy():
            logp = np.array([np.log(self.prior[i]) + np.sum(self.log_gaussian_density(i, f)) for i in range(self.count)])
            logp -= logp.max()
            p = np.exp(logp)
            salida.append(p / p.sum())
        return np.array(salida)

    def accuracy(self, y_test, y_pred):
        accuracy = np.sum(y_test == y_pred) / len(y_test)
        return accuracy

    def visualize(self, y_true, y_pred, target):

        tr = pd.DataFrame(data=y_true, columns=[target])
        pr = pd.DataFrame(data=y_pred, columns=[target])


        fig, ax = plt.subplots(1, 2, sharex='col', sharey='row', figsize=(15,6))

        sns.countplot(x=target, data=tr, ax=ax[0], palette='viridis', alpha=0.7, hue=target, dodge=False)
        sns.countplot(x=target, data=pr, ax=ax[1], palette='viridis', alpha=0.7, hue=target, dodge=False)


        fig.suptitle('True vs Predicted Comparison', fontsize=20)

        ax[0].tick_params(labelsize=12)
        ax[1].tick_params(labelsize=12)
        ax[0].set_title("True values", fontsize=18)
        ax[1].set_title("Predicted values", fontsize=18)
        plt.show()

---
## 🔍 Fase 2 · Comprensión de los datos — explorar la clase

Cargamos `spambase` (4 601 correos, 57 variables de frecuencia de palabras y símbolos).

In [ ]:
fig = px.histogram(data, x="spam", nbins=2, color="spam", labels={"spam": "Clase", "count": "Frecuencia"})
fig.update_layout(title_text="Distribución de Spam en el Dataset")
fig.show()


In [ ]:
data['spam'].value_counts()

---
## 🧹 Fase 3 · Preparación de los datos — estandarizar y dividir

Estandarizamos las variables y dividimos 70/30.

📚 Lectura recomendada: [9]

In [ ]:
# set features and target
X,y = data.iloc[:, :-1], data.iloc[:, -1]

le = preprocessing.LabelEncoder()
y =le.fit_transform(y)

#Hacerlo primero sin estandarizar magnitudes
# scale the data
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
X=pd.DataFrame(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True, random_state=42)

---
## 🤖 Fase 4 · Modelado — entrenar y predecir

Programamos Naive Bayes gaussiano desde cero (priori, medias, varianzas, log-posterior).

In [ ]:
x = NaiveBayesClassifier()
x.fit(X_train, y_train)

In [ ]:
predictions = x.predict(X_test)
x.accuracy(y_test, predictions)


In [ ]:
x.visualize(y_test, predictions, 'spam')

---
## 📊 Fase 5 · Evaluación

Evaluamos con exactitud, curva ROC (usando probabilidades), matriz de confusión, sensibilidad y especificidad.

📚 Lectura recomendada: [10] [11]

### **Curva ROC**

Curva ROC (acrónimo de Receiver Operating Characteristic, o Característica Operativa del Receptor) es una representación gráfica de la sensibilidad frente a la especificidad para un sistema clasificador binario según se varía el umbral de discriminación.

![Curvas ROC con su AUC y la línea de adivinar al azar](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_10.png)
*Figura: Curvas ROC con su AUC y la línea de adivinar al azar. [13]*

In [ ]:
#y_test=y_test.astype("category")
predictions = pd.DataFrame(predictions)
probas = x.predict_proba(X_test)[:, 1]   # probabilidad de spam; la curva ROC debe usar probabilidades, no etiquetas

In [ ]:
auc = roc_auc_score(y_test, probas)
print('AUC: %.2f' % auc)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, probas)

In [ ]:
plt.plot(fpr, tpr, color='orange', label='ROC')

plt.plot([0, 1], [0, 1], color='darkblue', linestyle='--')

plt.xlabel('Tasa de falsos positivos (1 − especificidad)')
plt.ylabel('Tasa de verdaderos positivos (sensibilidad)')
plt.title('Curva ROC del Naive Bayes (con probabilidades)')
plt.legend()
plt.show()

### **Matriz de confusion**

![Matriz de confusión: VP, FN, FP y VN](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_08.png)
*Figura: Matriz de confusión: VP, FN, FP y VN. [13]*

![Ejemplo de matriz de confusión con conteos reales](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_09.png)
*Figura: Ejemplo de matriz de confusión con conteos reales. [13]*

In [ ]:
matrix=confusion_matrix(y_test, predictions)
matrix

In [ ]:
fig = px.imshow(matrix, text_auto=True, x=["Predicho no spam", "Predicho spam"], y=["Real no spam", "Real spam"], color_continuous_scale="Blues")
fig.update_layout(title_text="Matriz de confusión")
fig.show()

\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}

In [ ]:
VN, FP, FN, VP = matrix.ravel()    # filas = real (0 = no spam, 1 = spam), columnas = predicho
print("Exactitud:", (VP + VN) / matrix.sum())

\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}

In [ ]:
print("Sensibilidad (spam detectado):", VP / (VP + FN))

\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}

In [ ]:
print("Especificidad (correo bueno bien clasificado):", VN / (VN + FP))

---
## 🧪 Refuerzo: Naive Bayes en la práctica: suavizado de Laplace y variantes

Con texto (como el correo electrónico) el supuesto "ingenuo" funciona sorprendentemente bien [5]. Dos detalles prácticos: (1) si una palabra nunca aparece en los correos *spam* de entrenamiento, su probabilidad sería 0 y anularía todo el producto; el **suavizado de Laplace** (parámetro `alpha`) suma un pequeño conteo a todas las palabras para evitarlo; (2) según cómo se representen las variables existen varias versiones: **Gaussiano** (valores continuos), **Multinomial** (conteos/frecuencias) y **Bernoulli** (presencia/ausencia) [7]. Comparamos las tres con validación cruzada y comprobamos que nuestro modelo "desde cero" coincide con el de `scikit-learn`.

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.model_selection import cross_val_score, StratifiedKFold

Xcrudo, ycrudo = data.iloc[:, :-1], data.iloc[:, -1]          # frecuencias de palabras sin estandarizar
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# 1) Nuestro modelo vs. scikit-learn en la misma partición
gnb = GaussianNB().fit(X_train, y_train)
print(f'Exactitud nuestro NB desde cero : {x.accuracy(y_test, x.predict(X_test)):.4f}')
print(f'Exactitud GaussianNB (sklearn)  : {gnb.score(X_test, y_test):.4f}')

# 2) Variantes de Naive Bayes
resultados = {
    'Gaussiano (variables estandarizadas)': cross_val_score(GaussianNB(), X, y, cv=cv).mean(),
    'Multinomial (frecuencias, alpha=1)':   cross_val_score(MultinomialNB(alpha=1.0), Xcrudo, ycrudo, cv=cv).mean(),
    'Bernoulli (palabra presente / ausente)': cross_val_score(BernoulliNB(), (Xcrudo > 0).astype(int), ycrudo, cv=cv).mean(),
}
print(pd.Series(resultados, name='Exactitud (5-fold)').round(4).to_string())

# 3) Efecto del suavizado de Laplace en el modelo multinomial
alphas = [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]
acc = [cross_val_score(MultinomialNB(alpha=a), Xcrudo, ycrudo, cv=cv).mean() for a in alphas]
plt.semilogx(alphas, acc, 'o-'); plt.xlabel('alpha (suavizado de Laplace)'); plt.ylabel('Exactitud (5-fold)'); plt.show()

### ✍️ Ejercicios
1. ¿Por qué el modelo Gaussiano rinde peor que el Multinomial o el Bernoulli en este conjunto? (Observe que las frecuencias de palabras no son normales: muchas son 0.)
2. ¿Qué pasa con `alpha` casi 0? ¿Y con `alpha` muy grande?
3. Calcule la **precisión** de la clase *spam* y explique por qué en un filtro real un falso positivo (correo bueno enviado a spam) es más costoso que un falso negativo.

---
## ✅ Conclusiones y siguientes pasos

**Correcciones importantes:** (1) la densidad gaussiana del modelo original tenía un factor 1/2 de más y se corrigió; (2) las fórmulas de sensibilidad y especificidad estaban intercambiadas con otras métricas; (3) la curva ROC se construyó con etiquetas en vez de probabilidades. Con estas correcciones el modelo desde cero coincide con `GaussianNB` de `scikit-learn`. Naive Bayes es rápido y sorprendentemente competitivo en texto [5], aunque sus probabilidades suelen estar mal calibradas [4].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [8].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Maron, M. E. (1961). Automatic indexing: An experimental inquiry. *Journal of the ACM, 8*(3), 404–417. https://doi.org/10.1145/321075.321084

**[4]** Domingos, P., & Pazzani, M. (1997). On the optimality of the simple Bayesian classifier under zero-one loss. *Machine Learning, 29*(2–3), 103–130. https://doi.org/10.1023/A:1007413511361

**[5]** McCallum, A., & Nigam, K. (1998). A comparison of event models for Naive Bayes text classification. *AAAI-98 Workshop on Learning for Text Categorization*, 41–48.

**[6]** Hopkins, M., Reeber, E., Forman, G., & Suermondt, J. (1999). *Spambase* [Conjunto de datos]. UCI Machine Learning Repository. https://archive.ics.uci.edu/dataset/94/spambase

**[7]** Murphy, K. P. (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press.

**[8]** Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction* (2.ª ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

**[9]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[10]** Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

**[11]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[12]** VanderPlas, J. (2016). *Python Data Science Handbook*. O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/ (figuras usadas desde https://github.com/jakevdp/PythonDataScienceHandbook)

**[13]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)